# Index_D Detective - The Ghost
Index_D sigue una señal oculta en otro índice. Este notebook la encuentra y la explota.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (14, 5)
sns.set_theme(style='darkgrid')

idx = pd.read_csv('../data/train_indices.csv', parse_dates=['Date'], index_col='Date')
print(idx.tail())

## 1. Búsqueda sistemática de la señal oculta

In [ ]:
results = []
other = [c for c in idx.columns if c != 'Index_D']

for col in other:
    for lag in range(-30, 31):  # negativo = Index_D va antes
        shifted = idx[col].shift(lag)
        corr_levels = idx['Index_D'].corr(shifted)
        corr_returns = idx['Index_D'].pct_change().corr(shifted.pct_change())
        results.append({'source': col, 'lag': lag, 'corr_levels': corr_levels, 'corr_returns': corr_returns})

res = pd.DataFrame(results)
print('TOP 15 por correlación en niveles:')
print(res.sort_values('corr_levels', ascending=False).head(15).to_string(index=False))

In [ ]:
# Identificar el mejor candidato
best = res.sort_values('corr_levels', ascending=False).iloc[0]
print(f"\nMejor candidato: {best['source']} con lag={best['lag']} (corr={best['corr_levels']:.4f})")

## 2. Visualización de la señal oculta

In [ ]:
source_col = best['source']
best_lag = int(best['lag'])

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(idx.index, idx['Index_D'], label='Index_D (real)', lw=1)
ax.plot(idx.index, idx[source_col].shift(best_lag), label=f'{source_col} lag={best_lag}', lw=1, alpha=0.8)
ax.set_title(f'Index_D vs {source_col} desfasado {best_lag} días')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Regresión lineal simple: ¿podemos predecir Index_D perfectamente?

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

df = idx[['Index_D', source_col]].copy()
df['source_shifted'] = df[source_col].shift(best_lag)
df.dropna(inplace=True)

X = df[['source_shifted']].values
y = df['Index_D'].values

lr = LinearRegression().fit(X, y)
y_pred = lr.predict(X)

print(f'R² = {r2_score(y, y_pred):.6f}')
print(f'RMSE = {np.sqrt(mean_squared_error(y, y_pred)):.4f}')
print(f'Coef: {lr.coef_[0]:.6f}  Intercept: {lr.intercept_:.4f}')

## 4. Exploración de transformaciones no-lineales

In [ ]:
# ¿Es una transformación más compleja?
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

for degree in [1, 2, 3]:
    pipe = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    pipe.fit(X, y)
    r2 = r2_score(y, pipe.predict(X))
    rmse = np.sqrt(mean_squared_error(y, pipe.predict(X)))
    print(f'Degree={degree}: R²={r2:.6f}, RMSE={rmse:.4f}')

## 5. Aprovechar en el modelo final

Si R² ≈ 1.0, podemos predecir Index_D directamente desde la señal fuente más el lag.
El lag positivo significa que **ya conocemos la señal fuente en el momento de predecir Index_D**
si el lag es positivo (source va antes que D).

In [ ]:
print(f"\nConclusion: Index_D se puede predecir usando {source_col} desfasado {best_lag} días.")
print(f"En predicción autorregresiva, cuando predicemos el día t de Index_D,")
if best_lag > 0:
    print(f"necesitamos {source_col} del día t-{best_lag}, que ya conocemos del histórico/predicciones anteriores.")
else:
    print(f"necesitamos {source_col} del día t+{abs(best_lag)}, que es una predicción futura todavía.")